<a href="https://colab.research.google.com/github/Vignesh-P-C/fake-news-detection-transformers/blob/main/07_extended_dataset_final_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U "transformers>=5.0.0" datasets accelerate scikit-learn


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
fake_df = pd.read_csv("Fake.csv", engine="python", on_bad_lines="skip")
fake_df["label"] = 0
fake_df["text"] = fake_df["title"].fillna("") + " " + fake_df["text"].fillna("")
fake_df = fake_df[["text", "label"]]

real_df = pd.read_csv("True.csv", engine="python", on_bad_lines="skip")
real_df["label"] = 1
real_df["text"] = real_df["title"].fillna("") + " " + real_df["text"].fillna("")
real_df = real_df[["text", "label"]]

print("Fake samples:", len(fake_df))
print("Real samples:", len(real_df))

In [ ]:
main_df = pd.concat([fake_df, real_df], axis=0, ignore_index=True)

main_df["text"] = main_df["text"].astype(str).str.strip()
main_df["label"] = main_df["label"].astype(int)
main_df = main_df.dropna(subset=["text", "label"])
main_df = main_df[main_df["text"].str.len() > 0]
main_df = main_df.drop_duplicates(subset=["text"])

print("Main dataset size:", len(main_df))
print(main_df["label"].value_counts())

In [ ]:
LIAR_COLS = [
    "id", "label", "statement", "subjects", "speaker",
    "job_title", "state", "party", "barely_true_count",
    "false_count", "half_true_count", "mostly_true_count",
    "pants_on_fire_count", "context"
]

FAKE_LABELS = {"pants-fire", "false", "barely-true"}
REAL_LABELS = {"mostly-true", "true"}

def load_liar(path):
    df = pd.read_csv(path, sep="\t", header=None, names=LIAR_COLS)
    df = df[["statement", "label"]].copy()
    df = df[df["label"].isin(FAKE_LABELS | REAL_LABELS)]
    df["label"] = df["label"].apply(lambda x: 0 if x in FAKE_LABELS else 1)
    df = df.rename(columns={"statement": "text"})
    df["text"] = df["text"].astype(str).str.strip()
    df = df.dropna(subset=["text"])
    df = df[df["text"].str.len() > 0]
    df = df.drop_duplicates(subset=["text"])
    df = df.reset_index(drop=True)
    return df

liar_train_df = load_liar("train.tsv")

print("LIAR train size (after dropping half-true):", len(liar_train_df))
print(liar_train_df["label"].value_counts())

In [ ]:
liar_oversampled = pd.concat([liar_train_df] * 4, ignore_index=True)

combined_df = pd.concat([main_df, liar_oversampled], axis=0, ignore_index=True)

combined_df["text"]  = combined_df["text"].astype(str).str.strip()
combined_df["label"] = combined_df["label"].astype(int)
combined_df = combined_df.dropna(subset=["text", "label"])
combined_df = combined_df[combined_df["text"].str.len() > 0]
combined_df = combined_df.drop_duplicates(subset=["text"])
combined_df = combined_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Combined dataset size:", len(combined_df))
print(combined_df["label"].value_counts())

In [ ]:
train_df, val_df = train_test_split(
    combined_df,
    test_size=0.1,
    stratify=combined_df["label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Train label dist:")
print(train_df["label"].value_counts())
print("Val label dist:")
print(val_df["label"].value_counts())

In [ ]:
train_dataset = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_dataset   = Dataset.from_pandas(val_df[["text", "label"]],   preserve_index=False)

train_dataset = train_dataset.rename_column("label", "labels")
val_dataset   = val_dataset.rename_column("label", "labels")

from datasets import ClassLabel
train_dataset = train_dataset.cast_column("labels", ClassLabel(names=["fake", "real"]))
val_dataset   = val_dataset.cast_column("labels",   ClassLabel(names=["fake", "real"]))

print("Train dataset:", train_dataset)
print("Val dataset:",   val_dataset)

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=["text"])
val_dataset   = val_dataset.map(tokenize,   batched=True, remove_columns=["text"])

if "token_type_ids" in train_dataset.column_names:
    train_dataset = train_dataset.remove_columns(["token_type_ids"])
if "token_type_ids" in val_dataset.column_names:
    val_dataset = val_dataset.remove_columns(["token_type_ids"])

train_dataset.set_format(type="torch")
val_dataset.set_format(type="torch")

print("Train columns:", train_dataset.column_names)
print("Val columns:",   val_dataset.column_names)

sample = train_dataset[0]
print("Sample keys:", list(sample.keys()))
print("input_ids shape:", sample["input_ids"].shape)
print("label:", sample["labels"].item())

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    problem_type="single_label_classification"
)

print("Model loaded. Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy":  round(acc, 4),
        "f1":        round(f1, 4),
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),

    logging_dir="./logs",
    logging_steps=100,
    report_to="none",

    seed=SEED,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
log_history = trainer.state.log_history
val_logs = [x for x in log_history if "eval_f1" in x]
best = val_logs[-1]

print("=== Validation Results ===")
for k, v in best.items():
    print(f"  {k}: {v}")

In [ ]:
test_df = load_liar("test.tsv")

print("Test set size:", len(test_df))
print(test_df["label"].value_counts())

In [ ]:
test_dataset = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)
test_dataset = test_dataset.rename_column("label", "labels")
test_dataset = test_dataset.cast_column("labels", ClassLabel(names=["fake", "real"]))

test_dataset = test_dataset.map(tokenize, batched=True, remove_columns=["text"])

if "token_type_ids" in test_dataset.column_names:
    test_dataset = test_dataset.remove_columns(["token_type_ids"])

test_dataset.set_format(type="torch")

print("Test dataset columns:", test_dataset.column_names)

In [ ]:
from torch.utils.data import DataLoader

def evaluate_model(model, dataset, batch_size=16):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    loader = DataLoader(dataset, batch_size=batch_size)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids    = batch["input_ids"].to(device)
            mask   = batch["attention_mask"].to(device)
            lbls   = batch["labels"].to(device)
            logits = model(input_ids=ids, attention_mask=mask).logits
            preds  = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
    acc  = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )
    return {"accuracy": round(acc,4), "precision": round(prec,4),
            "recall": round(rec,4), "f1": round(f1,4)}

torch.cuda.empty_cache()

val_results  = evaluate_model(model, val_dataset)
test_results = evaluate_model(model, test_dataset)

print("=== Validation Results ===")
for k, v in val_results.items():  print(f"  {k}: {v}")

print("\n=== Test Results ===")
for k, v in test_results.items(): print(f"  {k}: {v}")

In [ ]:
SAVE_DIR = "./news_credibility_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model and tokenizer saved to: {SAVE_DIR}")

In [ ]:
import torch.nn.functional as F

LABEL_MAP = {0: "FAKE", 1: "REAL"}

def predict_news(text: str) -> dict:
    model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=256
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        probs  = F.softmax(logits, dim=-1)[0].cpu().numpy()

    pred_id = int(np.argmax(probs))

    return {
        "label":       LABEL_MAP[pred_id],
        "confidence":  round(float(probs[pred_id]), 4),
        "prob_fake":   round(float(probs[0]), 4),
        "prob_real":   round(float(probs[1]), 4),
    }

In [ ]:
samples = [
    "Scientists confirm new vaccine is 99% effective in global trials.",
    "Government secretly replaced all birds with surveillance drones in 2001.",
    "The president signed the new infrastructure bill into law yesterday.",
]

for text in samples:
    result = predict_news(text)
    print(f"Text    : {text}")
    print(f"Result  : {result}")
    print()